<a href="https://colab.research.google.com/github/anirbanghoshsbi/.github.io/blob/master/work/err/cross_section_momentum/cross_sectional_momentum.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import duckdb
import numpy as np

In [3]:
# Reconnect and load data from DuckDB
conn = duckdb.connect("candle_data.duckdb")

# Read data into a Pandas DataFrame
df = conn.execute("SELECT * FROM candle_table").df()

# Close connection
conn.close()

# Display first few rows
print(df.shape)

(13164, 51)


In [5]:
df.tail(3)

,date,ADANIENT,ADANIPORTS,APOLLOHOSP,ASIANPAINT,AXISBANK,BAJAJ-AUTO,BAJFINANCE,BAJAJFINSV,BPCL,...,SUNPHARMA,TCS,TATACONSUM,TATAMOTORS,TATASTEEL,TECHM,TITAN,UPL,ULTRACEMCO,WIPRO
13161,2026-02-20,2160.800049,1511.500000,7615.5,2428.100098,1368.300049,9807.0,1030.199951,2058.500000,366.3,...,NaN,2686.199951,1156.199951,NaN,208.360001,1456.900024,4236.399902,752.349976,12766.0,209.860001
13162,2026-02-23,2191.000000,1555.800049,7693.0,2429.699951,1386.699951,9905.5,1031.000000,2051.600098,372.1,...,NaN,2676.300049,1171.800049,NaN,208.139999,1440.900024,4272.700195,644.799988,12976.0,205.889999
13163,2026-02-24,2167.000000,1548.800049,7680.5,2407.199951,1387.099976,9810.0,1021.400024,2042.699951,371.95,...,NaN,2576.100098,1173.099976,NaN,208.710007,1340.000000,4280.500000,628.750000,12949.0,200.179993


In [ ]:
df = df.copy()

df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date')
df = df.set_index('date')

In [8]:
# -----------------------------
# 1. Ensure proper date format
# -----------------------------

df = df.apply(pd.to_numeric, errors='coerce')
# -----------------------------
# 2. Calculate 6-Month Momentum
# 6 months ≈ 126 trading days
# -----------------------------
lookback = 126

momentum_6m = df.pct_change(periods=lookback)

# -----------------------------
# 3. Take Latest Date Snapshot
# -----------------------------
latest_momentum = momentum_6m.iloc[-1]

# Drop NaNs (stocks without sufficient history)
latest_momentum = latest_momentum.dropna()

# -----------------------------
# 4. Cross-Sectional Ranking
# -----------------------------
top_10 = (
    latest_momentum
    .sort_values(ascending=False)
    .head(10)
)

# -----------------------------
# 5. Display Results
# -----------------------------
result = pd.DataFrame({
    'symbol': top_10.index,
    'momentum_6m': top_10.values
})

print("Top 10 Nifty 50 Stocks by 6-Month Momentum")
print(result)

Top 10 Nifty 50 Stocks by 6-Month Momentum
       symbol  momentum_6m
0   POWERGRID     0.161420
1        SBIN     0.148378
2   EICHERMOT     0.136470
3  APOLLOHOSP     0.129319
4  ADANIPORTS     0.092397
5  BAJFINANCE     0.092231
6   SUNPHARMA     0.088529
7       TITAN     0.086504
8     DRREDDY     0.082306
9          LT     0.079407


/tmp/ipython-input-3025169071.py:12: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  momentum_6m = df.pct_change(periods=lookback)
